In [1]:
import pandas as pd
import os
import numpy as np
import re

/Users/tracyliu/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
pd.set_option('display.max_columns', None)

## Study: <span style="color:blue;">**Control**</span>

## A. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [3]:
file_path = "../Data/dicom_tag.xlsx"
dicom = pd.read_excel(file_path)

In [4]:
dicom[dicom["PatientID"]==4330431872]

,PatientID,AccessionNumber,PatientBirthDate,PatientAge,PatientSex,StudyDate,StudyTime,AcquisitionDate,Modality,Manufacturer,ManufacturerModelName,StudyDescription,SeriesNumber,SeriesDescription,Exposure,Rows,Columns,PixelSpacing,study,side
1235,4330431872,62499697,1974-07-01,044Y,F,2019-10-18,13:30:06,NaN,MG,"R2 Technology, Inc.",Cenova,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,SCREEN,NaN
1236,4330431872,62499697,1974-07-01,NaN,F,2019-10-18,13:30:06,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,71300000.0,L MLO Intelligent 2D,61.0,3328.0,2560.0,0.064613\0.064613,SCREEN,L
1237,4330431872,62499697,1974-07-01,NaN,F,2019-10-18,13:30:06,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,71300000.0,R MLO Intelligent 2D,52.0,3328.0,2560.0,0.065014\0.065014,SCREEN,R
1238,4330431872,62499697,1974-07-01,NaN,F,2019-10-18,13:30:06,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,71300000.0,R CC Intelligent 2D,54.0,3328.0,2560.0,0.065014\0.065014,SCREEN,R
1239,4330431872,62499697,1974-07-01,NaN,F,2019-10-18,13:30:06,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,71300000.0,L CC Intelligent 2D,61.0,3328.0,2560.0,0.064613\0.064613,SCREEN,L
1240,4330431872,70211453,1974-07-01,NaN,F,2017-10-13,08:52:52,NaN,PR,Philips Medical Systems,iSite Enterprise,MR BREAST WITH AND WITHOUT CONTRAST BILATERAL,1.0,59E5161F0,NaN,NaN,NaN,NaN,NaN,NaN
1241,4330431872,70211453,1974-07-01,NaN,F,2017-10-13,08:52:52,NaN,PR,Philips Medical Systems,iSite Enterprise,MR BREAST WITH AND WITHOUT CONTRAST BILATERAL,1.0,59E3C5C30,NaN,NaN,NaN,NaN,NaN,NaN
1242,4330431872,70211453,1974-07-01,NaN,F,2017-10-13,08:52:52,NaN,PR,Philips Medical Systems,iSite Enterprise,MR BREAST WITH AND WITHOUT CONTRAST BILATERAL,1.0,59E4A4250,NaN,NaN,NaN,NaN,NaN,NaN
1243,4330431872,70211453,1974-07-01,NaN,F,2017-10-13,08:52:54,NaN,PR,Philips Medical Systems,iSite Enterprise,MR BREAST WITH AND WITHOUT CONTRAST BILATERAL,1.0,59E517390,NaN,NaN,NaN,NaN,NaN,NaN
1244,4330431872,70211453,1974-07-01,NaN,F,2017-10-13,08:52:54,2017-10-13,MR,GE MEDICAL SYSTEMS,Signa HDxt,MR BREAST WITH AND WITHOUT CONTRAST BILATERAL,5.0,Ax Vibrant MULTIPHASE,NaN,512.0,512.0,0.5859\0.5859,NaN,L


In [5]:
dicom["PatientID"].unique().size

5515

In [6]:
PIDs = dicom["PatientID"].unique()

#### Create a list of unique DICOM entries to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

**shared parameters:**

DICOM ↔︎ EHR (pathology)

PatientID ↔︎ PATIENT_STUDY_ID 

AccessionNumber ↔︎ ACCESSION_NUMBER 

PatientBirthDate ↔︎ BIRTH_DATE

In [7]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER', 'PatientBirthDate': 'BIRTH_DATE'}, inplace=True)

In [8]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'BIRTH_DATE', 'PatientSex']
extract_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'StudyTime', 'study']
unique_dicom = dicom.drop_duplicates(subset=key_columns, keep='first')
unique_dicom = unique_dicom[extract_columns]
unique_dicom.reset_index(drop=True, inplace=True)

In [9]:
unique_dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study
0,4330018595,60103700,2020-06-02,09:10:52,DIAG
1,4330018595,60690108,2020-06-02,08:33:36,SCREEN
2,4330018595,63737104,2019-08-19,02:11:20,DIAG
3,4330029102,64888584,2019-05-16,08:32:09,SCREEN
4,4330044371,61647696,2020-02-06,15:37:19,SCREEN
...,...,...,...,...,...
15135,4339943307,63434181,2019-06-17,17:07:31,DIAG
15136,4339943307,63570279,2019-05-23,18:36:05,SCREEN
15137,4339945522,62906727,2019-10-18,03:00:54,DIAG
15138,4339945522,450264459,2022-10-14,11:54:03,NaN


## B. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [10]:
file_path = "../Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['pathology',
 'pathology_findings',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

---

## Study: <span style="color:blue;">**Control**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## 1. Merge with enteredit_findings 
* Columns: COMPOSITION_NAME, FINDING_LOCATION, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 1, 2

In [48]:
fn = file_name[6]
fn

'enteredit_findings'

### From **Control** folder

Find 26 matches in DICOM

In [79]:
study = "Control"

In [80]:
file_path = os.path.join("../Data/", study, "Cleaned", fn + ".xlsx")
BIRADS = pd.read_excel(file_path)

In [81]:
BIRADS_control = BIRADS[BIRADS['PATIENT_STUDY_ID'].isin(PIDs)]

In [82]:
BIRADS_control['PATIENT_STUDY_ID'].unique().size

26

### From **Cancer** folder

Find 5509 matches in DICOM

In [83]:
study = "Cancer"

In [84]:
file_path = os.path.join("../Data/", study, "Cleaned", fn + ".xlsx")
BIRADS = pd.read_excel(file_path)

In [85]:
BIRADS_cancer = BIRADS[BIRADS['PATIENT_STUDY_ID'].isin(PIDs)]

In [86]:
BIRADS_cancer['PATIENT_STUDY_ID'].unique().size

5509

In [87]:
set_control = set(BIRADS_control["PATIENT_STUDY_ID"].unique())
set_cancer = set(BIRADS_cancer["PATIENT_STUDY_ID"].unique())

set_duplicates = set_control & set_cancer
len(set_control), len(set_cancer), len(set_duplicates)

(26, 5509, 26)

In [88]:
duplicates = list(set_duplicates)

In [89]:
confirm_duplicates = []
for d in duplicates:
    tmp_control = BIRADS_control[BIRADS_control["PATIENT_STUDY_ID"]==d].reset_index(drop=True)
    tmp_cancer = BIRADS_cancer[BIRADS_cancer["PATIENT_STUDY_ID"]==d].reset_index(drop=True)
    if tmp_control.equals(tmp_cancer) == True:
        confirm_duplicates.append(d)

In [90]:
confirm_duplicates == duplicates

True

#### <span style="color:red;">**According to above, continue with BIRADS from cancer**</span>

In [91]:
BIRADS = BIRADS_cancer.copy()

### Merge dicom and enteredit_findings

In [122]:
unique_dicom[unique_dicom["PATIENT_STUDY_ID"]==4330431872]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study
136,4330431872,62499697,2019-10-18,13:30:06,SCREEN
137,4330431872,70211453,2017-10-13,08:52:52,NaN
138,4330431872,70212626,2017-10-13,17:08:49,DIAG
139,4330431872,76426245,2018-10-12,17:12:45,SCREEN


In [123]:
BIRADS[BIRADS["PATIENT_STUDY_ID"]==4330431872]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
757,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-03-30,1
758,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-03-30,1
759,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1
760,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1
761,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1
762,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1
763,4330431872,79589315,Heterogeneously dense (51% - 75%),NaN,99 - Post procedure mammograms for marker plac...,X-Follow-up post biopsy as directed by clinician,2017-10-31,1
764,4330431872,76426245,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-10-12,2
765,4330431872,65965175,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-02-01,1
766,4330431872,62499697,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-10-18,2


In [124]:
merge = pd.merge(
    BIRADS, 
    unique_dicom,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], 
    how='left'
)

merge['image_available'] = np.where(
    merge['StudyTime'].notna(), 
    1,  # 1 for Yes (match found)
    0   # 0 for No (no match found)
)

In [95]:
merge[merge["PATIENT_STUDY_ID"]==4330431872]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available
527,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-03-30,1,NaN,NaN,NaN,0
528,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-03-30,1,NaN,NaN,NaN,0
529,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1,2017-10-13,17:08:49,DIAG,1
530,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1,2017-10-13,17:08:49,DIAG,1
531,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1,2017-10-13,08:52:52,NaN,1
532,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1,2017-10-13,08:52:52,NaN,1
533,4330431872,79589315,Heterogeneously dense (51% - 75%),NaN,99 - Post procedure mammograms for marker plac...,X-Follow-up post biopsy as directed by clinician,2017-10-31,1,NaN,NaN,NaN,0
534,4330431872,76426245,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-10-12,2,2018-10-12,17:12:45,SCREEN,1
535,4330431872,65965175,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-02-01,1,NaN,NaN,NaN,0
536,4330431872,62499697,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-10-18,2,2019-10-18,13:30:06,SCREEN,1


In [194]:
merge["PATIENT_STUDY_ID"].unique().size

5509

## 2. Merge with pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Cancer: 
    * LESION_CLASS = 'Malignant'

In [97]:
fn = file_name[0]
fn

'pathology'

In [100]:
file_path = os.path.join("../Data/", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

### Store <span style="color:blue;">**control**</span> record only

In [170]:
yes_pathology = merge["PATIENT_STUDY_ID"].isin(pathology['PATIENT_STUDY_ID'])
no_pathology = ~merge["PATIENT_STUDY_ID"].isin(pathology['PATIENT_STUDY_ID'])

#### Option 1. Exclude patients having pathology records (N=299)

In [171]:
control_cohort = merge[no_pathology]
control_cohort.reset_index(inplace=True)

In [172]:
control_cohort.drop(["index"], axis=1, inplace=True)

<ipython-input-172-623f039a0749>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  control_cohort.drop(["index"], axis=1, inplace=True)


In [173]:
print("‼️ # of patients without pathology (control cohort):", control_cohort["PATIENT_STUDY_ID"].unique().size)
print("‼️ # of patients with pathology (exclude):", merge[yes_pathology]["PATIENT_STUDY_ID"].unique().size)

‼️ # of patients without pathology (control cohort): 299
‼️ # of patients with pathology (exclude): 5210


In [174]:
control_cohort[control_cohort["PATIENT_STUDY_ID"]==4330105691]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available
0,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-11-23,1,NaN,NaN,NaN,0
1,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2020-11-23,1,NaN,NaN,NaN,0
2,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,0
3,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,0
4,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,0
5,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,0
6,4330105691,451346206,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-18,2,NaN,NaN,NaN,0
7,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2022-07-18,1,2022-07-28,02:32:44,SCREEN,1
8,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-07-18,1,2022-07-28,02:32:44,SCREEN,1
9,4330105691,451963155,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-28,1,NaN,NaN,NaN,0


In [176]:
study = "Control"

In [177]:
output_file = os.path.join("../Data/", study, 'control_cohort' + ".xlsx")
control_cohort.to_excel(output_file, index=False)

<ipython-input-177-6a9fc139a31c>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  control_cohort.to_excel(output_file, index=False)


#### Option 2. Exclude patients having malignant findings (N=299+2866)

In [189]:
pathology_sort = pathology[pathology["PATIENT_STUDY_ID"].isin(merge['PATIENT_STUDY_ID'])].copy()
merge_sort = merge[yes_pathology].copy()

In [190]:
pathology_sort['PATHOLOGY_DATE'] = pd.to_datetime(pathology_sort['PATHOLOGY_DATE'])
merge_sort['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_sort['EXAM_COMPLETED_DATE'])

pathology_sort.sort_values(by =['PATIENT_STUDY_ID', 'PATHOLOGY_DATE'],inplace=True)
merge_sort.sort_values(by =['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'],inplace=True)

In [209]:
control_cohort = None
PIDs_without_pathology = []
PIDs_without_malignant = []
PIDs_with_malignant_high_risk = []

PIDs = merge["PATIENT_STUDY_ID"].unique()

for pid in PIDs:
    
    # merge_asof can only deal with one patient ID at a time, and the date needs to be sorted ascending
    
    path = pathology_sort[pathology_sort["PATIENT_STUDY_ID"]==pid]
    merg = merge_sort[merge_sort["PATIENT_STUDY_ID"]==pid]
    
    if path.size == 0:
        
        PIDs_without_pathology.append(pid)
    
    else:
    
        closest_match = pd.merge_asof(
        left=path,
        right=merg,
        left_on='PATHOLOGY_DATE',          # The date we are matching FROM
        right_on='EXAM_COMPLETED_DATE',    # The date we are matching TO
        by='PATIENT_STUDY_ID',             # The grouping key (Int64)
        direction='backward',             # Finds the closest date (after)
        allow_exact_matches=True,
        tolerance=pd.Timedelta(days=365)   # maximum time interval to be considered
        )
        
        
        # Check if LESION_CLASS = 'Malignant" OR "High Risk' ever exsits for the patient, if yes, skip
        column_to_check = 'LESION_CLASS' 
        closest_match[column_to_check] = closest_match[column_to_check].astype(str)
        lesion_terms = 'Malignant|High Risk'
        lesion_mask = closest_match[column_to_check].str.contains(lesion_terms, case=False, na=False, regex=True)
                
        
        if closest_match.loc[lesion_mask].size==0: 
            PIDs_without_malignant.append(pid)
            combine = merg.merge(closest_match, how='outer')
            control_cohort = pd.concat([control_cohort, combine])
        else:
            PIDs_with_malignant_high_risk.append(pid)

In [219]:
print("‼️ # of patients without pathology (control cohort):", len(PIDs_without_pathology))
print("‼️ # of patients with pathology (control cohort, no malignant):", control_cohort["PATIENT_STUDY_ID"].unique().size) # possible high risk
print("‼️ # of patients with pathology (exclude, malignant OR high risk):", len(PIDs_with_malignant_high_risk))

‼️ # of patients without pathology (control cohort): 299
‼️ # of patients with pathology (control cohort, no malignant): 2866
‼️ # of patients with pathology (exclude, malignant OR high risk): 2344


In [176]:
study = "Control"

In [212]:
output_file = os.path.join("../Data/", study, 'control_cohort_opt_2' + ".xlsx")
control_cohort.to_excel(output_file, index=False)

<ipython-input-212-c4332638c458>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  control_cohort.to_excel(output_file, index=False)


In [217]:
control_cohort[control_cohort["PATIENT_STUDY_ID"]==PIDs_without_malignant[2]]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330103113,79000052.0,Extremely dense (>75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-01-09,1.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN
1,4330103113,79000052.0,Extremely dense (>75%),NaN,1 - Negative,N-Normal interval follow-up,2018-01-09,1.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN
2,4330103113,79640396.0,Extremely dense (>75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-01-23,1.0,2018-01-23,13:19:33,DIAG,1.0,425798.0,2018-01-29,Benign,R
3,4330103113,61304464.0,Extremely dense (>75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-01-18,1.0,2020-01-18,09:36:41,SCREEN,1.0,NaN,NaT,NaN,NaN
4,4330103113,61304464.0,Extremely dense (>75%),NaN,1 - Negative,N-Normal interval follow-up,2020-01-18,1.0,2020-01-18,09:36:41,SCREEN,1.0,NaN,NaT,NaN,NaN
5,4330103113,61888889.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-02-12,1.0,2020-02-12,02:10:49,DIAG,1.0,NaN,NaT,NaN,NaN


## <span style="color:#FF6347;">**READ**</span> file

In [220]:
output_file = os.path.join("../Data/", "Control", 'control_cohort' + ".xlsx")
control_cohort = pd.read_excel(output_file)

In [221]:
control_PIDs = control_cohort["PATIENT_STUDY_ID"].unique()

In [224]:
control_cohort[control_cohort["PATIENT_STUDY_ID"]==control_PIDs[2]]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available
18,4330389368,76875097,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-28,2,2018-11-28,08:52:50,SCREEN,1
19,4330389368,62771201,Scattered fibroglandular (25% - 50%),NaN,1 - Negative,N-Normal interval follow-up,2019-12-19,2,NaN,NaN,NaN,0
20,4330389368,66490637,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2021-05-24,1,NaN,NaN,NaN,0
21,4330389368,66490637,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2021-05-24,1,NaN,NaN,NaN,0
22,4330389368,455543287,Heterogeneously dense (51% - 75%),NaN,4C - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-06-04,1,NaN,NaN,NaN,0
23,4330389368,455395411,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-23,1,NaN,NaN,NaN,0
24,4330389368,455395411,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-23,1,NaN,NaN,NaN,0
25,4330389368,454408109,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-05-26,2,NaN,NaN,NaN,0


## 3. Locate <span style="color:blue;">**control**</span> index year, "index status" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [223]:
# Create a column called "index status"
cohort = control_cohort.copy()
cohort['index status'] = None

In [31]:
# 1. Sort to find the earliest Malignant PATHOLOGY_DATE
cohort_sorted_pathology = cohort.sort_values(
    ['PATIENT_STUDY_ID', 'PATHOLOGY_DATE']
)

# Filter for only 'Malignant' records by dropping duplicates
first_malignant_indices = cohort_sorted_pathology[
    cohort_sorted_pathology['LESION_CLASS'] == 'Malignant'
].drop_duplicates(
    subset=['PATIENT_STUDY_ID'],
    keep='first'
).index.tolist()

# 2. Mark the corresponding "index status" as "INDEX"
cohort.loc[first_malignant_indices, 'index status'] = 'INDEX'

# Create a clean target DataFrame for the next merge step
target_df = cohort.loc[first_malignant_indices].copy().rename(
    columns={'PATHOLOGY_DATE': 'target pathology date'}
)

In [32]:
# 3. Locate Exams within One Year of Index Pathology Date

tar_lookup = target_df.set_index('PATIENT_STUDY_ID')['target pathology date']
cohort['target pathology date'] = cohort['PATIENT_STUDY_ID'].map(tar_lookup)

# Calculate the time difference (Pathology Date - Exam Date)
# Negative difference means the exam occurred AFTER the pathology (we want to exclude these later)
cohort['path to exam diff'] = cohort['target pathology date'] - cohort['EXAM_COMPLETED_DATE']

# Create a boolean mask for exams within 1 year (365 days) BEFORE or ON the pathology date
one_year = pd.Timedelta(days=365)
# Condition 1: Exam occurred before or on the pathology date (Pathology - Exam >= 0)
mask_preceding = cohort['path to exam diff'] >= pd.Timedelta(days=0)
# Condition 2: Exam occurred within 1 year before the pathology date (Pathology - Exam <= 365 days)
mask_within_year = cohort['path to exam diff'] <= one_year
# Combined mask: Preceding AND within 1 year
mask_index_cohort = mask_preceding & mask_within_year

In [33]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_cohort, 'index status'] = 'INDEX'

print(f"✅ Step 3 Complete. Marked {cohort[cohort['index status'] == 'INDEX'].shape[0]} rows as 'INDEX'.")

✅ Step 3 Complete. Marked 7040 rows as 'INDEX'.


In [34]:
cohort['index status'].unique()

array(['INDEX', None], dtype=object)

In [35]:
cohort[cohort["PATIENT_STUDY_ID"]==cancer_PIDs[8]]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,index status,target pathology date,path to exam diff
108,4330344296,71644307.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2.0,2017-07-31,02:23:29,SCREEN,1.0,NaN,NaT,NaN,NaN,None,2021-02-10,1290 days
109,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaT,NaN,NaN,None,2021-02-10,833 days
110,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaT,NaN,NaN,None,2021-02-10,833 days
111,4330344296,76695188.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1.0,2018-11-08,15:23:00,DIAG,1.0,NaN,NaT,NaN,NaN,None,2021-02-10,825 days
112,4330344296,62861703.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2.0,2019-12-20,02:44:43,SCREEN,1.0,NaN,NaT,NaN,NaN,None,2021-02-10,418 days
113,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaT,NaN,NaN,INDEX,2021-02-10,41 days
114,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaT,NaN,NaN,INDEX,2021-02-10,41 days
115,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,0.0,470168.0,2021-02-10,Malignant,R,INDEX,2021-02-10,16 days
116,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,0.0,470167.0,2021-02-10,Benign,R,INDEX,2021-02-10,16 days
117,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,0.0,478775.0,2021-03-05,Malignant,R,INDEX,2021-02-10,16 days


## 4. Locate <span style="color:blue;">**MO cancer**</span> year, "index status" = <span style="color:#00BFFF;">**INDEX-1**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

### 4a. Find the earliest <span style="color:#8A2BE2;">**INDEX**</span> exam date

Find the earliest EXAM_COMPLETED_DATE marked as <span style="color:#8A2BE2;">**"INDEX"**</span>

In [36]:
# 1. Isolate the current "INDEX" records
index_records = cohort[cohort['index status'] == 'INDEX'].copy()

# 2. Find the earliest EXAM_COMPLETED_DATE for the current "INDEX" cohort
earliest_index_exam = index_records.sort_values(
    ['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE']
).drop_duplicates(
    subset=['PATIENT_STUDY_ID'],
    keep='first'
)

# Rename the earliest exam date for clarity in the merge
earliest_index_exam.rename(
    columns={'EXAM_COMPLETED_DATE': 'target index date'},
    inplace=True
)

In [37]:
earliest_index_exam

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,target index date,duplicate_count,StudyDate,StudyTime,study,image_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,index status,target pathology date,path to exam diff
0,4330018595,63027507.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2020-06-05,315 days
13,4330066079,61259016.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,2.0,2020-01-14,18:25:58,DIAG,1.0,NaN,NaT,NaN,NaN,INDEX,2020-06-17,155 days
28,4330170072,79434021.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-27,1.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2017-11-16,20 days
40,4330303540,73123621.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-12-21,2.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2017-01-27,37 days
48,4330325599,75132824.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2016-04-20,1.0,2016-04-20,02:32:42,DIAG,1.0,NaN,NaT,NaN,NaN,INDEX,2016-04-20,0 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24795,4339508544,71405606.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2017-05-03,2.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2018-01-23,265 days
24816,4339577807,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,489989.0,2018-10-11,Malignant,L,INDEX,2018-10-11,NaT
24819,4339582412,78690585.0,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2018-04-25,2.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2018-05-08,13 days
24841,4339604670,68584905.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-11-27,1.0,NaN,NaN,NaN,0.0,NaN,NaT,NaN,NaN,INDEX,2020-12-22,25 days


### 4b. Find the preceding exam, <span style="color:blue;">**MO cancer**</span> year, <span style="color:#00BFFF;">**INDEX-1**</span>

Find the preceding exam that occurred at least 9 months before the earliest <span style="color:#8A2BE2;">**"INDEX"**</span> exam date but not before 18 months

In [38]:
# 3. Locate closest preceding exam, time difference between closest preceding exam date and Index Exam Date should fall in the range 9 months ~ 1.5 year

tar_lookup = earliest_index_exam.set_index('PATIENT_STUDY_ID')['target index date']
cohort['target index date'] = cohort['PATIENT_STUDY_ID'].map(tar_lookup)

# Calculate the time difference (INDEX Exam Date - Exam Date)
# Negative difference means the exam occurred AFTER the index (we want to exclude these later)
cohort['index to exam diff'] = cohort['target index date'] - cohort['EXAM_COMPLETED_DATE']


In [39]:
# Create a boolean mask for exams more than 9 months and less than 18 months BEFORE the index date
nine_month = pd.Timedelta(days=274)
eighteen_month = pd.Timedelta(days=548)
# Condition 1: Exam occurred before the INDEX exam date (INDEX Exam - Exam > 0)
mask_preceding = cohort['index to exam diff'] > pd.Timedelta(days=0)
# Condition 2: Exam occurred within or on 18 months before INDEX exam date (INDEX Exam - Exam <= 18 months)
mask_within_18_month = cohort['index to exam diff'] <= eighteen_month
# Condition 3: Exam occurred more than or on 9 months before INDEX exam date (INDEX Exam - Exam >= 9 months)
mask_more_9_month = cohort['index to exam diff'] >= nine_month
# Combined mask: Preceding AND within 18 months AND more than 9 months
mask_index_1_cohort = mask_preceding & mask_within_18_month & mask_more_9_month

In [40]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_1_cohort, 'index status'] = 'INDEX-1'

print(f"✅ Step 4 Complete. Marked {cohort[cohort['index status'] == 'INDEX-1'].shape[0]} rows as 'INDEX-1'.")

✅ Step 4 Complete. Marked 1198 rows as 'INDEX-1'.


In [41]:
cohort["index status"].unique()

array(['INDEX', None, 'INDEX-1'], dtype=object)

In [42]:
MO_cancer_cohort = None
MO_visit_only = None
PIDs_without_index_1 = []
PIDs_with_index_1 = []

PIDs = cancer_PIDs

for pid in PIDs:
    
    pid_series = cohort[cohort["PATIENT_STUDY_ID"]==pid]

    # Check if index status = 'INDEX-1' ever exsits for the patient, if none, skip
    column_to_check = 'index status' 
    index_terms = 'INDEX-1'
    both = pid_series[(pid_series[column_to_check]==index_terms) & (pid_series["image_available"]==1)]
    partial = pid_series[pid_series[column_to_check]==index_terms]


    if both.size==0 and partial.size==0: 
        PIDs_without_index_1.append(pid)
#         print("no INDEX-1")
    elif both.size==0 and partial.size!=0:
        PIDs_with_index_1.append(pid)
#         print("no image available")    
    else:
        PIDs_with_index_1.append(pid)
        MO_cancer_cohort = pd.concat([MO_cancer_cohort, pid_series])
        MO_visit_only = pd.concat([MO_visit_only, both])

In [43]:
MO_cancer_cohort.reset_index(inplace=True)
print("‼️ # of patients with cancer:",len(cancer_PIDs))
print("‼️ # of patients without INDEX-1:", len(PIDs_without_index_1))
print("‼️ # of patients with INDEX-1:", len(PIDs_with_index_1))
print("‼️ # of patients with with INDEX-1 and available image:",MO_cancer_cohort["PATIENT_STUDY_ID"].unique().size)

‼️ # of patients with cancer: 1956
‼️ # of patients without INDEX-1: 1045
‼️ # of patients with INDEX-1: 911
‼️ # of patients with with INDEX-1 and available image: 384


In [44]:
MO_cancer_cohort.columns

Index(['index', 'PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME',
       'FINDING_LOCATION', 'FINDING_CATEGORY', 'FINDING_REC',
       'EXAM_COMPLETED_DATE', 'duplicate_count', 'StudyDate', 'StudyTime',
       'study', 'image_available', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS',
       'SIDE', 'index status', 'target pathology date', 'path to exam diff',
       'target index date', 'index to exam diff'],
      dtype='object')

In [46]:
MO_cancer_cohort.drop(["index"], axis=1, inplace=True)

In [47]:
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort' + ".xlsx")
MO_cancer_cohort.to_excel(output_file, index=False)

<ipython-input-47-2649940d5efd>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  MO_cancer_cohort.to_excel(output_file, index=False)


## <span style="color:#FF6347;">**READ**</span> file

In [ ]:
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort' + ".xlsx")
MO_cancer_cohort = pd.read_excel(output_file)